In [36]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta


In [37]:
data_path='raw_datasets/'
eth_regs=pd.read_csv(f'{data_path}eth_domains_registrations.csv').drop(['registrant_address','registration_date'], axis=1)

In [38]:
eth_regs=eth_regs.rename(columns={'labelhash_hex':'labelhash'})
eth_regs

,domain_name,price,namehash,labelhash
0,rilxxlir.eth,0.007863,0x14f992cdd302644816a275e88fea2816741a571484b5...,0x00000425b4462e19460bedb4bccfcf16d270975ef882...
1,hansberry.eth,0.003712,0x66356dd6a49c090de63e29f72f5a6cadebadbae856e2...,0x000008a51dfcb3e27040cf86337c239b16af1fb6be15...
2,lalegend.eth,0.002886,0x55c674315e9b7a5cbd991667b12e6fc96506da61b395...,0x00000a84132c08f14b780b230921d313076f87449f9a...
3,peterwagner.eth,0.001976,0xd474ef6c09fae479d64f2c680f435c538c214bbef31d...,0x00000d2d44db55b19c072cd2928d3e91486e21b3533c...
4,stateland.eth,0.000544,0x5bc36a91cba700c6e2ff606642835e805a1b0d2a01a1...,0x0000140a7d7816a16b879ca9b9a1f86320b4e20c69da...
...,...,...,...,...
3490611,gangster-all-star.eth,0.000481,0x3bed2f4ced170882ea466224649e75f72e818ce90a2a...,0xffffedf12801036e130308427a774f3ad2d80cd7f55f...
3490612,chain-feeds.eth,0.002754,0xcd8de7e94071f6ca3d9824768f6f0b815acbee85df62...,0xfffff14cddea9a5ca1f18af1c04934199602b2ca48d2...
3490613,68435.eth,0.009944,0x9532931df53722d16330a4e59e1093eae0fb2fc89a83...,0xfffff82baf00574f83c7cf968a045c80f61b7a2f233c...
3490614,prismai-deployer.eth,0.001257,0x201ce495cc1f762e4d49ec472759d95ced2377bb3865...,0xfffffd0249d3699bbe66975ed19c4b989d892611d1e4...


In [39]:
eth_to_usd=pd.read_csv(f'{data_path}ETHUSDT_1d.csv')
eth_to_usd['open_time_ms'] = pd.to_datetime(eth_to_usd['open_time_ms'], unit='ms')
eth_price=eth_to_usd.sort_values(by='open_time_ms', ascending=False).iloc[0]['open_price']

In [40]:
eth_regs['eth_reg_price']=np.round(eth_regs['price']*eth_price,2)

Переходим к историям транзакций

In [ ]:
eth_sales_base=pd.read_csv(f'{data_path}eth_domains_sales_base.csv').drop(['tx_hash','unix_timestamp'], axis=1)
eth_sales_wrap=pd.read_csv(f'{data_path}eth_domains_sales_wrapped.csv').drop(['tx_hash','unix_timestamp'], axis=1)

У врап версии есть значения price_usd = "nil" 

In [58]:
eth_sales_wrap['price_usd'].value_counts()

price_usd
<nil>                  64
0.21477150000000003    21
0.21519400000000002    20
0.2148805              16
0.09892691             15
                       ..
19.07388                1
0.642309                1
84.33959999999999       1
1815.9066000000003      1
434.77719               1
Name: count, Length: 6213, dtype: int64

In [59]:
eth_sales_wrap[eth_sales_wrap['price_usd']=='<nil>']

,namehash,price_usd
107,0x91c316fa0b10eb933b4bb27860b0cb0ce281543762d3...,<nil>
130,0xedf7367e757e14301deb9a0f979ac26b6c85f3b23b43...,<nil>
133,0x944bd13afd24277e0ab6c9df8d772e14495461cd1f69...,<nil>
141,0x5b6b32a76f75ce057e6030933d0833eed799edaf957e...,<nil>
197,0xfeccea6d15f066f722c1077524d5660c4579993b359b...,<nil>
...,...,...
3520,0xe1d45d863f195ffef9c31b00c60c9c0fdab1d34a8c54...,<nil>
3528,0xf390a2540daa56d5cbbb3fd1cb8466ee131eed169f60...,<nil>
3535,0xb90ff2a8cbdc878c0c6ef41c4181a157cbe38856e250...,<nil>
3536,0xb90ff2a8cbdc878c0c6ef41c4181a157cbe38856e250...,<nil>


In [69]:
eth_sales_wrap=eth_sales_wrap[eth_sales_wrap['price_usd']!='<nil>']
eth_sales_wrap['price_usd']=eth_sales_wrap['price_usd'].astype(float)

## Чё делать с ними пока хз

In [70]:
eth_sales_base['namehash'] = eth_sales_base['labelhash'].map(eth_regs.set_index('labelhash')['namehash'])

In [71]:
eth_sales_base['namehash'].isna().sum()

np.int64(16)

Объединяем списки транзакций

In [72]:
eth_sales=pd.concat([eth_sales_wrap,
eth_sales_base.drop('labelhash', axis=1)], ignore_index=True
)

In [74]:
eth_sales_agg= (
    eth_sales.groupby('namehash', as_index=False).agg(
        eth_sales_count=('price_usd', 'count'),
        eth_sales_volume_usd=('price_usd', 'sum'),
        eth_last_sale_price_usd=('price_usd', 'last'),
    )
)

In [87]:
eth_res=eth_regs.merge(eth_sales_agg, on='namehash', how='left')


In [ ]:
Откидываем eth из domain_name

In [89]:
eth_res['domain_name']=eth_res['domain_name'].apply( lambda x : x.split(".")[0] )
eth_res

,domain_name,price,namehash,labelhash,eth_reg_price,eth_sales_count_x,eth_sales_volume_usd_x,eth_last_sale_price_usd_x,eth_sales_count_y,eth_sales_volume_usd_y,eth_last_sale_price_usd_y
0,rilxxlir,0.007863,0x14f992cdd302644816a275e88fea2816741a571484b5...,0x00000425b4462e19460bedb4bccfcf16d270975ef882...,17.22,NaN,NaN,NaN,NaN,NaN,NaN
1,hansberry,0.003712,0x66356dd6a49c090de63e29f72f5a6cadebadbae856e2...,0x000008a51dfcb3e27040cf86337c239b16af1fb6be15...,8.13,NaN,NaN,NaN,NaN,NaN,NaN
2,lalegend,0.002886,0x55c674315e9b7a5cbd991667b12e6fc96506da61b395...,0x00000a84132c08f14b780b230921d313076f87449f9a...,6.32,NaN,NaN,NaN,NaN,NaN,NaN
3,peterwagner,0.001976,0xd474ef6c09fae479d64f2c680f435c538c214bbef31d...,0x00000d2d44db55b19c072cd2928d3e91486e21b3533c...,4.33,NaN,NaN,NaN,NaN,NaN,NaN
4,stateland,0.000544,0x5bc36a91cba700c6e2ff606642835e805a1b0d2a01a1...,0x0000140a7d7816a16b879ca9b9a1f86320b4e20c69da...,1.19,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
3490611,gangster-all-star,0.000481,0x3bed2f4ced170882ea466224649e75f72e818ce90a2a...,0xffffedf12801036e130308427a774f3ad2d80cd7f55f...,1.05,NaN,NaN,NaN,NaN,NaN,NaN
3490612,chain-feeds,0.002754,0xcd8de7e94071f6ca3d9824768f6f0b815acbee85df62...,0xfffff14cddea9a5ca1f18af1c04934199602b2ca48d2...,6.03,NaN,NaN,NaN,NaN,NaN,NaN
3490613,68435,0.009944,0x9532931df53722d16330a4e59e1093eae0fb2fc89a83...,0xfffff82baf00574f83c7cf968a045c80f61b7a2f233c...,21.78,1.0,38.41376,38.41376,1.0,38.41376,38.41376
3490614,prismai-deployer,0.001257,0x201ce495cc1f762e4d49ec472759d95ced2377bb3865...,0xfffffd0249d3699bbe66975ed19c4b989d892611d1e4...,2.75,NaN,NaN,NaN,NaN,NaN,NaN


In [90]:
sol_sales

,unix_timestamp,domain_name,usd_price
0,1626022811,dab,25.00
1,1626044830,tiddernips,20.10
2,1626068681,hello1,20.10
3,1626073517,3commas,51.00
4,1626084013,privacy,50.10
...,...,...,...
272356,1775736120,qrpay,2059.25
272357,1775736120,inspire,2059.25
272358,1775736208,insight,4118.50
272359,1775736208,sovrin,4118.50


То же самое с sol 

In [91]:
sol_sales=pd.read_csv(f'{data_path}sol_domains_sales.csv').drop(['tx_signature','domain_key','bidder_key','unix_timestamp'], axis=1)
sol_sales.drop('successful', axis=1, inplace=True) # нет неудачных записей

eth_sales_agg = (
    sol_sales.groupby('domain_name', as_index=False)
    .agg(
        sol_sales_count=('usd_price', 'count'),
        sol_sales_volume_usd=('usd_price', 'sum'),
        sol_last_sale_price_usd=('usd_price', 'last'),
    )
)


In [93]:
result_df=eth_res.merge(eth_sales_agg, on='domain_name', how='left')

In [96]:
len(result_df[result_df.notnull().all(axis=1)])

8839